In [1]:
from fixture.factory.dataset.ohlcv import factory_ohlcv_cycle

df = factory_ohlcv_cycle()
df

Date,Open,High,Low,Close,Volume
datetime[μs],f64,f64,f64,f64,i64
2000-01-01 00:00:00,101.0,101.22,99.52,99.71,46913
2000-01-02 00:00:00,99.71,99.92,99.3,99.61,37069
2000-01-03 00:00:00,99.61,99.91,99.19,99.4,36107
2000-01-04 00:00:00,98.41,99.9,98.23,99.62,16237
2000-01-05 00:00:00,99.62,100.77,99.61,100.56,42774
…,…,…,…,…,…
2000-04-05 00:00:00,108.58,108.65,107.39,107.57,29540
2000-04-06 00:00:00,107.57,108.77,107.31,108.65,45414
2000-04-07 00:00:00,108.65,108.8,108.0,108.12,42624


In [2]:
from feature.closes.service import derive_closes


closes = derive_closes(df, 8)
closes

Date,now,lag_1,lag_2,lag_3,lag_4,lag_5,lag_6,lag_7,lag_8
datetime[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64
2000-01-10 00:00:00,37.357495,18.731213,5.922417,70.349559,0.994382,93.916166,22.10834,-21.104475,-10.034117
2000-01-11 00:00:00,14.708048,37.357495,18.731213,5.922417,70.349559,0.994382,93.916166,22.10834,-21.104475
2000-01-12 00:00:00,-31.403362,14.708048,37.357495,18.731213,5.922417,70.349559,0.994382,93.916166,22.10834
2000-01-13 00:00:00,-67.061395,-31.403362,14.708048,37.357495,18.731213,5.922417,70.349559,0.994382,93.916166
2000-01-14 00:00:00,71.974676,-67.061395,-31.403362,14.708048,37.357495,18.731213,5.922417,70.349559,0.994382
…,…,…,…,…,…,…,…,…,…
2000-04-05 00:00:00,5.579319,32.608218,208.389984,-57.00728,33.21474,56.241509,76.768442,-12.515044,139.51011
2000-04-06 00:00:00,99.899083,5.579319,32.608218,208.389984,-57.00728,33.21474,56.241509,76.768442,-12.515044
2000-04-07 00:00:00,-48.899853,99.899083,5.579319,32.608218,208.389984,-57.00728,33.21474,56.241509,76.768442


In [3]:
from feature.closes.derive import derive_closes_n4

closes_n4 = derive_closes_n4(df)
closes_n4

Date,now,lag_1,lag_2,lag_3,lag_4
datetime[μs],f64,f64,f64,f64,f64
2000-01-06 00:00:00,0.994382,93.916166,22.10834,-21.104475,-10.034117
2000-01-07 00:00:00,70.349559,0.994382,93.916166,22.10834,-21.104475
2000-01-08 00:00:00,5.922417,70.349559,0.994382,93.916166,22.10834
2000-01-09 00:00:00,18.731213,5.922417,70.349559,0.994382,93.916166
2000-01-10 00:00:00,37.357495,18.731213,5.922417,70.349559,0.994382
…,…,…,…,…,…
2000-04-05 00:00:00,5.579319,32.608218,208.389984,-57.00728,33.21474
2000-04-06 00:00:00,99.899083,5.579319,32.608218,208.389984,-57.00728
2000-04-07 00:00:00,-48.899853,99.899083,5.579319,32.608218,208.389984


In [4]:
closes.equals(closes_n4)

False

In [5]:
# closesの特徴量分析

import plotly.express as px
import plotly.graph_objects as go
import polars as pl

# 相関行列ヒートマップ
corr = closes.drop("Date").to_pandas().corr()
fig = px.imshow(
    corr,
    labels=dict(x="Features", y="Features", color="Correlation"),
    x=corr.columns,
    y=corr.columns,
    title="特徴量間の相関関係ヒートマップ",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1
)
fig.update_layout(width=800, height=800)
fig.show()

# 特徴量分布のボックスプロット
fig = go.Figure()
for col in closes.columns[1:]:  # Dateを除外
    fig.add_trace(go.Box(
        y=closes[col],
        name=col,
        boxpoints='outliers',
        jitter=0.3,
        pointpos=-1.8
    ))
fig.update_layout(
    title="特徴量の分布と外れ値",
    yaxis_title="Value",
    boxmode='group'
)
fig.show()

# 時系列プロット（now特徴量）
fig = px.line(
    closes.to_pandas(),
    x="Date",
    y="now",
    title="'now'特徴量の時系列変化",
    labels={"now": "Value"},
    template="plotly_white"
)
# 移動平均を計算してプロット
fig.add_trace(go.Scatter(
    x=closes["Date"],
    y=closes["now"].rolling_mean(5),
    mode="lines",
    name="5日移動平均",
    line=dict(color="red", dash="dot")
))
fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Value",
    hovermode="x unified"
)
fig.show()

# ラグ特徴量の相互作用（3D散布図）
fig = px.scatter_3d(
    closes.head(100).to_pandas(),
    x='lag_1',
    y='lag_2',
    z='now',
    color='lag_3',
    title="ラグ特徴量の3D相互作用",
    labels={'lag_1': 'Lag 1', 'lag_2': 'Lag 2', 'now': 'Current'},
    color_continuous_scale=px.colors.sequential.Viridis
)
fig.update_layout(
    scene=dict(
        xaxis_title='Lag 1',
        yaxis_title='Lag 2',
        zaxis_title='Current Value'
    ),
    width=1000,
    height=800
)
fig.show()